In [3]:
from lightning import LightningModule, Trainer
from torch import nn, Tensor
import torch
from typing import List, Tuple
from torchmetrics import Accuracy
from lightning.pytorch import LightningDataModule
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path
import os

class MNISTLightningDataModule(LightningDataModule):
    def __init__(
        self, root: Path, batch_size: int, num_workers: int, val_fraction: float = 0.1
    ):
        super().__init__()
        self.root = root
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_fraction = val_fraction
        self.transform = transforms.ToTensor()
        self._train_dataset = None
        self._val_dataset = None
        self._test_dataset = None

    def prepare_data(self) -> None:  # type: ignore[override]
        datasets.MNIST(root=self.root, train=True, download=True)
        datasets.MNIST(root=self.root, train=False, download=True)

    def setup(self, stage: str | None = None) -> None:  # type: ignore[override]
        if stage == "fit" or stage is None:
            full_train = datasets.MNIST(
                root=self.root,
                train=True,
                download=False,
                transform=self.transform,
            )
            val_size = int(len(full_train) * self.val_fraction)
            train_size = len(full_train) - val_size
            self._train_dataset, self._val_dataset = random_split(
                full_train,
                [train_size, val_size],
                generator=torch.Generator().manual_seed(42),
            )
        if stage == "test" or stage is None:
            self._test_dataset = datasets.MNIST(
                root=self.root,
                train=False,
                download=False,
                transform=self.transform,
            )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self._train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self._val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self._test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )


class DeepReLULightningModule(LightningModule):
    def __init__(
        self,
        input_dim: int,
        width: int,
        depth: int,
        dropout: float,
        batchnorm: bool,
        lr: float,
        momentum: float,
        l2_penalty: float,
    ):
        super().__init__()
        self.save_hyperparameters()
        if depth < 1:
            raise ValueError("Depth must be >= 1")
        layers: List[nn.Module] = [nn.Flatten(), nn.Linear(input_dim, width), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            if batchnorm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(width, 10))
        self.network = nn.Sequential(*layers)
        self.criterion = nn.CrossEntropyLoss()
        self.train_accuracy = Accuracy(task="multiclass", num_classes=10)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=10)
        self.test_accuracy = Accuracy(task="multiclass", num_classes=10)

    def forward(self, x: Tensor) -> Tensor:  # type: ignore[override]
        return self.network(x)

    def _compute_l2_penalty(self) -> Tensor:
        coefficient: float = float(self.hparams.l2_penalty)
        if coefficient <= 0:
            return torch.zeros((), device=self.device)
        penalty = torch.zeros((), device=self.device)
        for param in self.parameters():
            if param.requires_grad:
                penalty = penalty + param.pow(2).sum()
        return 0.5 * coefficient * penalty

    def training_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        if float(self.hparams.l2_penalty) > 0:
            l2_penalty = self._compute_l2_penalty()
            loss = loss + l2_penalty
            self.log(
                "train/l2_penalty_epoch",
                l2_penalty,
                on_step=False,
                on_epoch=True,
                prog_bar=False,
            )
        preds = logits.argmax(dim=1)
        acc = self.train_accuracy(preds, labels)
        self.log("train/loss_epoch", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(
            "train/accuracy_epoch", acc, on_step=False, on_epoch=True, prog_bar=True
        )
        return loss

    def validation_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = self.val_accuracy(preds, labels)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/accuracy", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch: Tuple[Tensor, Tensor], batch_idx: int) -> Tensor:  # type: ignore[override]
        inputs, labels = batch
        logits = self(inputs)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = self.test_accuracy(preds, labels)
        self.log("test/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test/accuracy", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self) -> None:  # type: ignore[override]
        self.train_accuracy.reset()

    def on_validation_epoch_end(self) -> None:  # type: ignore[override]
        self.val_accuracy.reset()

    def on_test_epoch_end(self) -> None:  # type: ignore[override]
        self.test_accuracy.reset()

    def configure_optimizers(self):  # type: ignore[override]
        optimizer = torch.optim.SGD(
            self.parameters(),
            lr=self.hparams.lr,
            momentum=self.hparams.momentum,
        )
        # reduce lr by a factor of 10 every at epochs 60, 100, and 200
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[60, 100, 200],
            gamma=0.1,
        )
        return [optimizer], [scheduler]

In [ ]:
# train a model and then estimate the L2 regularization strength with an InductiveBiasEstimator

input_dim = 28 * 28
width = 256
depth = 3
dropout = 0.2
batchnorm = True
lr = 0.01
momentum = 0.9
l2_penalty = 1e-4
model = DeepReLULightningModule(
    input_dim=input_dim,
    width=width,
    depth=depth,
    dropout=dropout,
    batchnorm=batchnorm,
    lr=lr,
    momentum=momentum,
    l2_penalty=l2_penalty,
)
trainer = Trainer(
    max_epochs=5, 
    accelerator="auto", 
    devices=1
    )
train_dm = MNISTLightningDataModule(
    root=os.path.expanduser("~/inductive-bias/MNIST"),
    batch_size=64,
    num_workers=4,
)
train_dm.prepare_data()
train_dm.setup("fit")
trainer.fit(model, train_dm)


Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name           | Type               | Params | Mode 
--------------------------------------------------------------
0 | network        | Sequential         | 336 K  | train
1 | criterion      | CrossEntropyLoss   | 0      | train
2 | train_accuracy | MulticlassAccuracy | 0      | train
3 | val_accuracy   | MulticlassAccuracy | 0      | train
4 | test_accuracy  | MulticlassAccuracy | 0      | train
--------------------------------------------------------------
336 K     Trainable params
0         Non-trainable params
336 K     Total params
1.345     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined